Курсовая работа Витрик Григорий ИД24-1 2.3

Анализ характеристик музыкальных треков для поиска паттернов популярности и жанровой принадлежности

In [1]:
import os
import pandas as pd
import numpy as np
import requests
import xml.etree.ElementTree as ET

from collections import Counter
from sklearn.preprocessing import MultiLabelBinarizer

#MTG-Jamendo

In [2]:
!git clone -q https://github.com/MTG/mtg-jamendo-dataset.git

In [3]:
%cd mtg-jamendo-dataset

/content/mtg-jamendo-dataset


In [4]:
!ls data

autotagging_genre.tsv	    download			     raw.tsv
autotagging_instrument.tsv  raw_30s_cleantags_50artists.tsv  splits
autotagging_moodtheme.tsv   raw_30s_cleantags.tsv	     tag_map.json
autotagging_top50tags.tsv   raw_30s.tsv			     tags
autotagging.tsv		    raw.meta.tsv


In [19]:
def read_mtg_file(filename, tag_prefix, column_name):
    rows = []

    with open(filename, "r", encoding="utf-8") as f:
        for line in f:
            if line.startswith("#"):
                continue

            parts = line.strip().split("\t")
            rows.append(parts)

    data = []

    for parts in rows[1:]:
        tags = [
            tag.replace(tag_prefix, "")
            for tag in parts[5:]
            if tag.startswith(tag_prefix)
        ]

        data.append({
            "track_id": parts[0],
            "artist_id": parts[1],
            "album_id": parts[2],
            "path": parts[3],
            "duration": float(parts[4]),
            column_name: tags
        })

    return pd.DataFrame(data)

In [20]:
df_instruments = read_mtg_file(
    "data/autotagging_instrument.tsv",
    "instrument---",
    "instruments"
)

df_instruments.head()

,track_id,artist_id,album_id,path,duration,instruments
0,track_0000382,artist_000020,album_000046,82/382.mp3,211.1,[voice]
1,track_0000383,artist_000020,album_000046,83/383.mp3,113.1,[voice]
2,track_0000384,artist_000020,album_000046,84/384.mp3,115.7,[voice]
3,track_0000386,artist_000020,album_000046,86/386.mp3,103.4,[voice]
4,track_0000387,artist_000020,album_000046,87/387.mp3,257.1,[voice]


In [21]:
df_genres = read_mtg_file(
    "data/autotagging_genre.tsv",
    "genre---",
    "genres"
)

df_genres.head()

,track_id,artist_id,album_id,path,duration,genres
0,track_0000214,artist_000014,album_000031,14/214.mp3,124.6,[punkrock]
1,track_0000215,artist_000014,album_000031,15/215.mp3,151.4,[metal]
2,track_0000216,artist_000014,album_000031,16/216.mp3,234.9,[metal]
3,track_0000217,artist_000014,album_000031,17/217.mp3,127.9,[punkrock]
4,track_0000218,artist_000014,album_000031,18/218.mp3,180.7,[punkrock]


In [22]:
df_mood = read_mtg_file(
    "data/autotagging_moodtheme.tsv",
    "mood/theme---",
    "moodtheme"
)

df_mood.head()

,track_id,artist_id,album_id,path,duration,moodtheme
0,track_0000948,artist_000087,album_000149,48/948.mp3,212.7,[background]
1,track_0000950,artist_000087,album_000149,50/950.mp3,248.0,[background]
2,track_0000951,artist_000087,album_000149,51/951.mp3,199.7,[background]
3,track_0002165,artist_000326,album_000347,65/2165.mp3,229.0,[film]
4,track_0002263,artist_000320,album_000366,63/2263.mp3,494.7,[melancholic]


In [23]:
print("Инструменты:", df_instruments.shape)
print("Жанры:", df_genres.shape)
print("Mood/theme:", df_mood.shape)

Инструменты: (25135, 6)
Жанры: (55215, 6)
Mood/theme: (18486, 6)


In [24]:
df_genres.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55215 entries, 0 to 55214
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   track_id   55215 non-null  object 
 1   artist_id  55215 non-null  object 
 2   album_id   55215 non-null  object 
 3   path       55215 non-null  object 
 4   duration   55215 non-null  float64
 5   genres     55215 non-null  object 
dtypes: float64(1), object(5)
memory usage: 2.5+ MB


Объединение датасетов


In [25]:
df_tracks = df_genres.merge(
    df_instruments[["track_id", "instruments"]],
    on="track_id",
    how="inner"
)

In [26]:
df_tracks = df_tracks.merge(
    df_mood[["track_id", "moodtheme"]],
    on="track_id",
    how="left"
)

In [27]:
df_tracks.head()

,track_id,artist_id,album_id,path,duration,genres,instruments,moodtheme
0,track_0000382,artist_000020,album_000046,82/382.mp3,211.1,[classical],[voice],NaN
1,track_0000383,artist_000020,album_000046,83/383.mp3,113.1,[classical],[voice],NaN
2,track_0000384,artist_000020,album_000046,84/384.mp3,115.7,[classical],[voice],NaN
3,track_0000386,artist_000020,album_000046,86/386.mp3,103.4,[classical],[voice],NaN
4,track_0000387,artist_000020,album_000046,87/387.mp3,257.1,[classical],[voice],NaN


In [28]:
print("Строк:", df_tracks.shape[0])
print("Столбцов:", df_tracks.shape[1])

print("Уникальных треков:", df_tracks["track_id"].nunique())
print("Уникальных исполнителей:", df_tracks["artist_id"].nunique())
print("Уникальных альбомов:", df_tracks["album_id"].nunique())

Строк: 24894
Столбцов: 8
Уникальных треков: 24894
Уникальных исполнителей: 1990
Уникальных альбомов: 5671


In [29]:
df_tracks[
    [
        "track_id",
        "duration",
        "genres",
        "instruments",
        "moodtheme"
    ]
].head(10)

,track_id,duration,genres,instruments,moodtheme
0,track_0000382,211.1,[classical],[voice],NaN
1,track_0000383,113.1,[classical],[voice],NaN
2,track_0000384,115.7,[classical],[voice],NaN
3,track_0000386,103.4,[classical],[voice],NaN
4,track_0000387,257.1,[classical],[voice],NaN
5,track_0000388,124.6,[classical],[voice],NaN
6,track_0000389,150.8,[classical],[voice],NaN
7,track_0000390,133.7,[classical],[voice],NaN
8,track_0000391,82.1,[classical],[voice],NaN
9,track_0000392,127.3,[classical],[voice],NaN


In [30]:
df_tracks.isna().sum()

,0
track_id,0
artist_id,0
album_id,0
path,0
duration,0
genres,0
instruments,0
moodtheme,14378


In [32]:
print(
    "Повторяющихся track_id:",
    df_tracks["track_id"].duplicated().sum())

Повторяющихся track_id: 0


In [33]:
df_tracks[["duration"]].describe()

,duration
count,24894.000000
mean,234.487820
std,148.679291
min,30.000000
25%,157.500000
50%,216.000000
75%,280.800000
max,3966.200000


In [34]:
print("Наиболее частые жанры:")
display(
    df_tracks["genres"]
    .explode()
    .value_counts()
    .head(10)
)

print("Наиболее частые инструменты:")
display(
    df_tracks["instruments"]
    .explode()
    .value_counts()
    .head(10)
)

print("Наиболее частые mood/theme:")
display(
    df_tracks["moodtheme"]
    .explode()
    .value_counts()
    .head(10)
)

Наиболее частые жанры:


,count
genres,
electronic,5572
soundtrack,4441
ambient,4005
classical,3812
pop,3305
rock,3203
chillout,2258
easylistening,2147
jazz,1625


Наиболее частые инструменты:


,count
instruments,
piano,7630
synthesizer,7442
drums,6094
bass,5698
guitar,4760
electricguitar,4229
computer,2073
keyboard,2036
acousticguitar,1931


Наиболее частые mood/theme:


,count
moodtheme,
happy,889
energetic,868
emotional,864
film,844
relaxing,842
melodic,762
dark,658
dream,540
sad,538


In [39]:
df_tracks_clean = df_tracks.copy()

In [40]:
df_tracks_clean = df_tracks_clean[
    df_tracks_clean["duration"] > 0]

df_tracks_clean["moodtheme"] = df_tracks_clean["moodtheme"].apply(
    lambda x: x if isinstance(x, list) else [])

df_tracks_clean.isna().sum()

,0
track_id,0
artist_id,0
album_id,0
path,0
duration,0
genres,0
instruments,0
moodtheme,0


In [52]:
rows_meta = []

with open("data/raw.meta.tsv", "r", encoding="utf-8") as f:
    for line in f:
        parts = line.rstrip("\n").split("\t")
        rows_meta.append(parts)

rows_meta[:3]

[['TRACK_ID',
  'ARTIST_ID',
  'ALBUM_ID',
  'TRACK_NAME',
  'ARTIST_NAME',
  'ALBUM_NAME',
  'RELEASEDATE',
  'URL'],
 ['track_0000214',
  'artist_000014',
  'album_000031',
  'Intro chiante',
  'David TMX',
  'Pouce !',
  '2004-12-28',
  'http://www.jamendo.com/track/214'],
 ['track_0000215',
  'artist_000014',
  'album_000031',
  'Aide toi...',
  'David TMX',
  'Pouce !',
  '2004-12-28',
  'http://www.jamendo.com/track/215']]

In [59]:
from collections import Counter

lengths = Counter(len(row) for row in rows_meta)

lengths

Counter({8: 56638, 9: 2})

In [60]:
bad_rows = [
    row for row in rows_meta[1:]
    if len(row) != len(rows_meta[0])
]

print("Проблемных строк:", len(bad_rows))

for row in bad_rows[:10]:
    print(len(row), row)

Проблемных строк: 2
9 ['track_1081574', 'artist_362862', 'album_128886', "Non hai più cuore (Hip Hop R'n'b Mix)", '', 'SLEEYO', "COMPRENSIONE INTRINSECA ''y2007 the original''", '2013-11-22', 'http://www.jamendo.com/track/1081574']
9 ['track_1081578', 'artist_362862', 'album_128886', 'Quando sto con te (Funky Hip Hop Mix)', '', 'SLEEYO', "COMPRENSIONE INTRINSECA ''y2007 the original''", '2013-11-22', 'http://www.jamendo.com/track/1081578']


In [61]:
good_rows = [
    row for row in rows_meta[1:]
    if len(row) == len(rows_meta[0])
]

bad_rows = [
    row for row in rows_meta[1:]
    if len(row) != len(rows_meta[0])
]

print("Всего строк:", len(rows_meta) - 1)
print("Корректных строк:", len(good_rows))
print("Удалено некорректных строк:", len(bad_rows))

Всего строк: 56639
Корректных строк: 56637
Удалено некорректных строк: 2


In [62]:
df_meta = pd.DataFrame(
    good_rows,
    columns=rows_meta[0]
)

df_meta.head()

,TRACK_ID,ARTIST_ID,ALBUM_ID,TRACK_NAME,ARTIST_NAME,ALBUM_NAME,RELEASEDATE,URL
0,track_0000214,artist_000014,album_000031,Intro chiante,David TMX,Pouce !,2004-12-28,http://www.jamendo.com/track/214
1,track_0000215,artist_000014,album_000031,Aide toi...,David TMX,Pouce !,2004-12-28,http://www.jamendo.com/track/215
2,track_0000216,artist_000014,album_000031,Une charogne,David TMX,Pouce !,2004-12-28,http://www.jamendo.com/track/216
3,track_0000217,artist_000014,album_000031,Aimons nous...,David TMX,Pouce !,2004-12-28,http://www.jamendo.com/track/217
4,track_0000218,artist_000014,album_000031,Jouvencelle,David TMX,Pouce !,2004-12-28,http://www.jamendo.com/track/218


In [63]:
df_meta = df_meta.rename(columns={
    "TRACK_ID": "track_id",
    "ARTIST_ID": "artist_id",
    "ALBUM_ID": "album_id",
    "TRACK_NAME": "track_name",
    "ARTIST_NAME": "artist_name",
    "ALBUM_NAME": "album_name",
    "RELEASEDATE": "release_date",
    "URL": "url"
})

df_meta.head()

,track_id,artist_id,album_id,track_name,artist_name,album_name,release_date,url
0,track_0000214,artist_000014,album_000031,Intro chiante,David TMX,Pouce !,2004-12-28,http://www.jamendo.com/track/214
1,track_0000215,artist_000014,album_000031,Aide toi...,David TMX,Pouce !,2004-12-28,http://www.jamendo.com/track/215
2,track_0000216,artist_000014,album_000031,Une charogne,David TMX,Pouce !,2004-12-28,http://www.jamendo.com/track/216
3,track_0000217,artist_000014,album_000031,Aimons nous...,David TMX,Pouce !,2004-12-28,http://www.jamendo.com/track/217
4,track_0000218,artist_000014,album_000031,Jouvencelle,David TMX,Pouce !,2004-12-28,http://www.jamendo.com/track/218


In [64]:
df_meta.isna().sum()

,0
track_id,0
artist_id,0
album_id,0
track_name,0
artist_name,0
album_name,0
release_date,0
url,0


In [65]:
df_tracks_clean = df_tracks_clean.merge(
    df_meta[
        [
            "track_id",
            "track_name",
            "artist_name",
            "album_name",
            "release_date",
            "url"
        ]
    ],
    on="track_id",
    how="left"
)

In [66]:
df_tracks_clean[
    [
        "track_id",
        "track_name",
        "artist_name",
        "genres",
        "instruments",
        "moodtheme"
    ]
].head(10)

,track_id,track_name,artist_name,genres,instruments,moodtheme
0,track_0000382,Soon and Very Soon (Negro Spiritual),Les Petits Chanteurs de Montigny,[classical],[voice],[]
1,track_0000383,Go tell it on the mountain (Negro Spiritual),Les Petits Chanteurs de Montigny,[classical],[voice],[]
2,track_0000384,Let us break bread (Negro Spiritual),Les Petits Chanteurs de Montigny,[classical],[voice],[]
3,track_0000386,Cantate Domino (Johann Crüger),Les Petits Chanteurs de Montigny,[classical],[voice],[]
4,track_0000387,Cantique de Jean Racine (Gabriel Fauré),Les Petits Chanteurs de Montigny,[classical],[voice],[]
5,track_0000388,Jubilemus Exultemus (François Couperin),Les Petits Chanteurs de Montigny,[classical],[voice],[]
6,track_0000389,Nocturnes (W.A. Mozart),Les Petits Chanteurs de Montigny,[classical],[voice],[]
7,track_0000390,Psaume 61 (Joseph Haydn),Les Petits Chanteurs de Montigny,[classical],[voice],[]
8,track_0000391,Cantate Domino (G.O. Pitoni),Les Petits Chanteurs de Montigny,[classical],[voice],[]
9,track_0000392,Messe à 3 voix (Antonio Lotti),Les Petits Chanteurs de Montigny,[classical],[voice],[]


In [69]:
print(
    "Треков с названием:",
    df_tracks_clean["track_name"].notna().sum(),
    "из",
    len(df_tracks_clean)
)

print(
    "Треков с исполнителем:",
    df_tracks_clean["artist_name"].notna().sum(),
    "из",
    len(df_tracks_clean)
)

Треков с названием: 24894 из 24894
Треков с исполнителем: 24894 из 24894


#XML

In [70]:
df_sample = df_tracks_clean.sample(n=300, random_state=42).copy()

In [71]:
def get_musicbrainz_info(artist, track):
    url = "https://musicbrainz.org/ws/2/recording/"

    params = {
        "query": f'recording:"{track}" AND artist:"{artist}"',
        "fmt": "xml",
        "limit": 1
    }

    headers = {
        "User-Agent": "MusicCoursework/1.0"
    }

    response = requests.get(
        url,
        params=params,
        headers=headers
    )

    root = ET.fromstring(response.text)

    ns = {
        "m": "http://musicbrainz.org/ns/mmd-2.0#"
    }

    recording = root.find(
        ".//m:recording",
        ns
    )

    if recording is None:
        return None, None

    recording_id = recording.attrib.get("id")

    length = recording.find(
        "m:length",
        ns
    )

    if length is not None:
        length_ms = int(length.text)
    else:
        length_ms = None

    return recording_id, length_ms

In [74]:
import time

musicbrainz_results = []

for _, row in df_sample.iterrows():

    recording_id, length_ms = get_musicbrainz_info(
        row["artist_name"],
        row["track_name"]
    )

    musicbrainz_results.append({
        "track_id": row["track_id"],
        "recording_id": recording_id,
        "musicbrainz_length_ms": length_ms
    })

    time.sleep(1)

In [76]:
df_musicbrainz = pd.DataFrame(
    musicbrainz_results
)

found = df_musicbrainz["recording_id"].notna().sum()

print("Всего проверено:", len(df_musicbrainz))
print("Найдено в MusicBrainz:", found)

print(
    "Процент найденных:",
    round(found / len(df_musicbrainz) * 100, 2),
    "%"
)

Всего проверено: 300
Найдено в MusicBrainz: 57
Процент найденных: 19.0 %


In [81]:
df_sample = df_tracks_clean.sample(
    n=300,
    random_state=42
).copy()

In [82]:
df_sample.columns

Index(['track_id', 'artist_id', 'album_id', 'path', 'duration', 'genres',
       'instruments', 'moodtheme', 'track_name', 'artist_name', 'album_name',
       'release_date', 'url'],
      dtype='object')

In [83]:
df_sample = df_sample.merge(
    df_musicbrainz,
    on="track_id",
    how="left"
)

In [84]:
df_sample.columns

Index(['track_id', 'artist_id', 'album_id', 'path', 'duration', 'genres',
       'instruments', 'moodtheme', 'track_name', 'artist_name', 'album_name',
       'release_date', 'url', 'recording_id', 'musicbrainz_length_ms'],
      dtype='object')

In [85]:
df_sample[
    [
        "track_id",
        "track_name",
        "artist_name",
        "genres",
        "instruments",
        "recording_id",
        "musicbrainz_length_ms"
    ]
].head(20)

,track_id,track_name,artist_name,genres,instruments,recording_id,musicbrainz_length_ms
0,track_1299924,patience,magagog,[techno],[synthesizer],None,NaN
1,track_1415331,Easy Cheesy (ID 405),Lobo Loco,"[dance, easylistening]",[piano],None,NaN
2,track_1247120,Propaganda mon amour,Joseph Curwen,[experimental],[synthesizer],None,NaN
3,track_1151251,FACETS 3,CŒDES Pierre-Marie,"[alternative, newage, orchestral]",[computer],None,NaN
4,track_1113990,Lifeless Devotion,Yusuke Tsutsumi,"[ambient, classical]","[cello, drums, piano, strings, viola, violin, ...",0e9ec173-8cc9-4b12-9158-38773936540c,276000.0
5,track_1221863,Blame,The A.J. Gatz Project,"[alternative, pop, singersongwriter]","[guitar, piano]",None,NaN
6,track_1255090,piano piece 052,Ton-Kunst,"[classical, improvisation, soundtrack]",[piano],None,NaN
7,track_1163235,Floating Isle - Vermillion,Floating Isle,"[electronic, pop]","[drums, electricguitar, synthesizer]",None,NaN
8,track_1150348,Eole génome,Julien Boulier,[electronic],"[computer, piano, strings, synthesizer]",d024e6ba-2797-4188-9ee1-f2f34f832f48,246516.0
9,track_0715557,A fine slice of emptiness,Arnoldsrecords,"[lounge, newage, soundtrack]","[acousticguitar, pipeorgan]",None,NaN


In [86]:
df_sample["musicbrainz_duration"] = (
    df_sample["musicbrainz_length_ms"] / 1000
)

In [87]:
df_sample[
    [
        "track_name",
        "duration",
        "musicbrainz_duration"
    ]
].head(20)

,track_name,duration,musicbrainz_duration
0,patience,233.5,NaN
1,Easy Cheesy (ID 405),235.7,NaN
2,Propaganda mon amour,246.9,NaN
3,FACETS 3,322.3,NaN
4,Lifeless Devotion,276.1,276.000
5,Blame,330.0,NaN
6,piano piece 052,162.5,NaN
7,Floating Isle - Vermillion,194.3,NaN
8,Eole génome,246.5,246.516
9,A fine slice of emptiness,210.2,NaN


#API

In [90]:
LASTFM_API_KEY = "секретик"

In [91]:
def get_lastfm_info(artist, track):
    url = "https://ws.audioscrobbler.com/2.0/"

    params = {
        "method": "track.getInfo",
        "api_key": LASTFM_API_KEY,
        "artist": artist,
        "track": track,
        "format": "json"
    }

    response = requests.get(url, params=params)
    data = response.json()

    if "track" not in data:
        return None, None, []

    track_data = data["track"]

    listeners = int(track_data.get("listeners", 0))
    playcount = int(track_data.get("playcount", 0))

    tags = [
        tag["name"]
        for tag in track_data.get("toptags", {}).get("tag", [])
    ]

    return listeners, playcount, tags

In [92]:
lastfm_results = []

for _, row in df_sample.iterrows():

    listeners, playcount, tags = get_lastfm_info(
        row["artist_name"],
        row["track_name"]
    )

    lastfm_results.append({
        "track_id": row["track_id"],
        "listeners": listeners,
        "playcount": playcount,
        "lastfm_tags": tags
    })

In [94]:
df_lastfm = pd.DataFrame(lastfm_results)

found_lastfm = df_lastfm["listeners"].notna().sum()

print("Всего проверено:", len(df_lastfm))
print("Найдено в Last.fm:", found_lastfm)

print(
    "Процент найденных:",
    round(found_lastfm / len(df_lastfm) * 100, 2),
    "%"
)

Всего проверено: 300
Найдено в Last.fm: 189
Процент найденных: 63.0 %


In [105]:
df_sample = df_sample.merge(
    df_lastfm,
    on="track_id",
    how="left"
)

In [106]:
df_sample.columns

Index(['track_id', 'artist_id', 'album_id', 'path', 'duration', 'genres',
       'instruments', 'moodtheme', 'track_name', 'artist_name', 'album_name',
       'release_date', 'url', 'recording_id', 'musicbrainz_length_ms',
       'musicbrainz_duration', 'listeners', 'playcount', 'lastfm_tags'],
      dtype='object')

In [107]:
df_sample[
    [
        "track_name",
        "artist_name",
        "genres",
        "instruments",
        "recording_id",
        "listeners",
        "playcount",
        "lastfm_tags"
    ]
].head(20)

,track_name,artist_name,genres,instruments,recording_id,listeners,playcount,lastfm_tags
0,patience,magagog,[techno],[synthesizer],None,NaN,NaN,[]
1,Easy Cheesy (ID 405),Lobo Loco,"[dance, easylistening]",[piano],None,1.0,2.0,[]
2,Propaganda mon amour,Joseph Curwen,[experimental],[synthesizer],None,NaN,NaN,[]
3,FACETS 3,CŒDES Pierre-Marie,"[alternative, newage, orchestral]",[computer],None,NaN,NaN,[]
4,Lifeless Devotion,Yusuke Tsutsumi,"[ambient, classical]","[cello, drums, piano, strings, viola, violin, ...",0e9ec173-8cc9-4b12-9158-38773936540c,218.0,790.0,[]
5,Blame,The A.J. Gatz Project,"[alternative, pop, singersongwriter]","[guitar, piano]",None,12.0,41.0,[]
6,piano piece 052,Ton-Kunst,"[classical, improvisation, soundtrack]",[piano],None,NaN,NaN,[]
7,Floating Isle - Vermillion,Floating Isle,"[electronic, pop]","[drums, electricguitar, synthesizer]",None,2.0,3.0,[]
8,Eole génome,Julien Boulier,[electronic],"[computer, piano, strings, synthesizer]",d024e6ba-2797-4188-9ee1-f2f34f832f48,NaN,NaN,[]
9,A fine slice of emptiness,Arnoldsrecords,"[lounge, newage, soundtrack]","[acousticguitar, pipeorgan]",None,15.0,21.0,[]


In [108]:
df_sample.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 19 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   track_id               300 non-null    object 
 1   artist_id              300 non-null    object 
 2   album_id               300 non-null    object 
 3   path                   300 non-null    object 
 4   duration               300 non-null    float64
 5   genres                 300 non-null    object 
 6   instruments            300 non-null    object 
 7   moodtheme              300 non-null    object 
 8   track_name             300 non-null    object 
 9   artist_name            300 non-null    object 
 10  album_name             300 non-null    object 
 11  release_date           300 non-null    object 
 12  url                    300 non-null    object 
 13  recording_id           57 non-null     object 
 14  musicbrainz_length_ms  55 non-null     float64
 15  musicb

In [109]:
df_sample.isna().sum()

,0
track_id,0
artist_id,0
album_id,0
path,0
duration,0
genres,0
instruments,0
moodtheme,0
track_name,0
artist_name,0


In [110]:
df_sample[
    ["listeners", "playcount"]
].describe()

,listeners,playcount
count,189.000000,189.000000
mean,74.185185,291.492063
std,228.924806,926.231639
min,1.000000,1.000000
25%,2.000000,5.000000
50%,8.000000,24.000000
75%,29.000000,124.000000
max,2088.000000,9834.000000


#Очистка


In [113]:
df_cleaning = df_sample.copy()
print(
    "Повторяющихся track_id:",
    df_cleaning["track_id"].duplicated().sum()
)

Повторяющихся track_id: 0


In [114]:
df_cleaning = df_cleaning[
    df_cleaning["duration"] > 0]

numeric_columns = [
    "duration",
    "musicbrainz_length_ms",
    "musicbrainz_duration",
    "listeners",
    "playcount"
]

for column in numeric_columns:
    df_cleaning[column] = pd.to_numeric(
        df_cleaning[column],
        errors="coerce")

In [116]:
list_columns = [
    "genres",
    "instruments",
    "moodtheme",
    "lastfm_tags"
]

for column in list_columns:
    df_cleaning[column] = df_cleaning[column].apply(
        lambda x: x if isinstance(x, list) else []
    )

In [117]:
df_popularity = df_cleaning.dropna(
    subset=["listeners", "playcount"]
).copy()

In [119]:
print("Полный объединённый набор:", len(df_cleaning))
print("Для анализа популярности:", len(df_popularity))

Полный объединённый набор: 300
Для анализа популярности: 189


In [120]:
df_cleaning.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 19 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   track_id               300 non-null    object 
 1   artist_id              300 non-null    object 
 2   album_id               300 non-null    object 
 3   path                   300 non-null    object 
 4   duration               300 non-null    float64
 5   genres                 300 non-null    object 
 6   instruments            300 non-null    object 
 7   moodtheme              300 non-null    object 
 8   track_name             300 non-null    object 
 9   artist_name            300 non-null    object 
 10  album_name             300 non-null    object 
 11  release_date           300 non-null    object 
 12  url                    300 non-null    object 
 13  recording_id           57 non-null     object 
 14  musicbrainz_length_ms  55 non-null     float64
 15  musicb

In [123]:
df_cleaning = df_cleaning.dropna(
    subset=[
        "recording_id",
        "musicbrainz_length_ms",
        "listeners",
        "playcount"
    ]
).copy()

df_cleaning.isna().sum()

,0
track_id,0
artist_id,0
album_id,0
path,0
duration,0
genres,0
instruments,0
moodtheme,0
track_name,0
artist_name,0


In [124]:
df_cleaning.shape

(51, 19)

In [122]:
df_popularity[
    [
        "track_name",
        "artist_name",
        "genres",
        "instruments",
        "listeners",
        "playcount"
    ]
].head(10)

,track_name,artist_name,genres,instruments,listeners,playcount
1,Easy Cheesy (ID 405),Lobo Loco,"[dance, easylistening]",[piano],1.0,2.0
4,Lifeless Devotion,Yusuke Tsutsumi,"[ambient, classical]","[cello, drums, piano, strings, viola, violin, ...",218.0,790.0
5,Blame,The A.J. Gatz Project,"[alternative, pop, singersongwriter]","[guitar, piano]",12.0,41.0
7,Floating Isle - Vermillion,Floating Isle,"[electronic, pop]","[drums, electricguitar, synthesizer]",2.0,3.0
9,A fine slice of emptiness,Arnoldsrecords,"[lounge, newage, soundtrack]","[acousticguitar, pipeorgan]",15.0,21.0
11,Secret Headquarters,The Eighth,"[90s, hiphop]",[beat],11.0,19.0
12,Brave Heart,SONIC MYSTERY,"[ambient, drumnbass, easylistening, electronic...","[bell, drummachine, drums, synthesizer]",107.0,871.0
13,Prelude 17,Josh Woodward,"[ambient, classical, newage]",[piano],4.0,16.0
14,Yagé,SaReGaMa,"[experimental, soundtrack, world]",[flute],156.0,568.0
15,Apocalypse now,Stefano mocini,"[choir, orchestral, soundtrack]","[orchestra, piano, strings]",21.0,140.0


#Векторизация трека

In [127]:
audio_path = "/content/City of stars.flac"

In [130]:
import librosa


y, sr = librosa.load(
    audio_path,
    sr=None
)

print("Частота дискретизации:", sr)
print("Количество отсчётов:", len(y))
print("Длительность фрагмента:", len(y) / sr)

Частота дискретизации: 44100
Количество отсчётов: 6602064
Длительность фрагмента: 149.70666666666668


In [131]:
mfcc = librosa.feature.mfcc(
    y=y,
    sr=sr,
    n_mfcc=13
)

spectral_centroid = librosa.feature.spectral_centroid(
    y=y,
    sr=sr
)

zero_crossing_rate = librosa.feature.zero_crossing_rate(y)

rms = librosa.feature.rms(y=y)

tempo, _ = librosa.beat.beat_track(
    y=y,
    sr=sr)

In [132]:
audio_vector = {
    "tempo": float(np.asarray(tempo).item()),
    "spectral_centroid": float(spectral_centroid.mean()),
    "zero_crossing_rate": float(zero_crossing_rate.mean()),
    "rms": float(rms.mean())
}

for i in range(13):
    audio_vector[f"mfcc_{i + 1}"] = float(
        mfcc[i].mean()
    )

In [133]:
df_audio_vector = pd.DataFrame(
    [audio_vector]
)

df_audio_vector

,tempo,spectral_centroid,zero_crossing_rate,rms,mfcc_1,mfcc_2,mfcc_3,mfcc_4,mfcc_5,mfcc_6,mfcc_7,mfcc_8,mfcc_9,mfcc_10,mfcc_11,mfcc_12,mfcc_13
0,109.956782,1948.787024,0.036128,0.079209,-289.081146,157.831345,17.481144,13.448689,-2.367397,10.625193,-3.008813,-0.128517,-3.716826,-5.685863,-1.025713,-2.030943,-8.394639


#Сохранение

In [134]:
df_cleaning.to_parquet(
    "music_data_merged.parquet",
    index=False
)

In [135]:
df_check = pd.read_parquet(
    "music_data_merged.parquet"
)

df_check.head()

,track_id,artist_id,album_id,path,duration,genres,instruments,moodtheme,track_name,artist_name,album_name,release_date,url,recording_id,musicbrainz_length_ms,musicbrainz_duration,listeners,playcount,lastfm_tags
0,track_1113990,artist_442477,album_133060,90/1113990.mp3,276.1,"[ambient, classical]","[cello, drums, piano, strings, viola, violin, ...","[melancholic, slow]",Lifeless Devotion,Yusuke Tsutsumi,Suicide Beauty Girl,2014-03-25,http://www.jamendo.com/track/1113990,0e9ec173-8cc9-4b12-9158-38773936540c,276000.0,276.000,218.0,790.0,[]
1,track_0206047,artist_341323,album_030556,47/206047.mp3,168.0,"[ambient, drumnbass, easylistening, electronic...","[bell, drummachine, drums, synthesizer]","[adventure, documentary, meditative]",Brave Heart,SONIC MYSTERY,HOPE REMAINS,2008-08-26,http://www.jamendo.com/track/206047,66bcaad6-8c5c-4dd6-af1f-80fa4df912c7,168000.0,168.000,107.0,871.0,[]
2,track_0053499,artist_004812,album_006123,99/53499.mp3,150.0,"[experimental, soundtrack, world]",[flute],[],Yagé,SaReGaMa,Spirits of Ancestors (EP),2007-06-17,http://www.jamendo.com/track/53499,5eb54b39-f96b-4a36-9ff2-91689a9046bf,137363.0,137.363,156.0,568.0,[]
3,track_1374955,artist_008236,album_161658,55/1374955.mp3,252.8,"[metal, progressive, rock]",[guitar],[],Law 1: You Cannot Win,Daniel Bautista,The Pink Album,2016-09-10,http://www.jamendo.com/track/1374955,f3d9e521-9e96-400b-993a-03030e0b3977,252741.0,252.741,121.0,537.0,[]
4,track_1109890,artist_441505,album_132604,90/1109890.mp3,183.8,"[electronic, indie]","[bass, drums, synthesizer]",[advertising],Swimming,Podington Bear,Background,2014-03-14,http://www.jamendo.com/track/1109890,6366a9db-741b-47f0-8513-db10db5a7070,183000.0,183.000,26.0,60.0,[]


In [136]:
def update_data(old_data, new_data, key):
    combined = pd.concat(
        [old_data, new_data],
        ignore_index=True
    )

    combined = combined.drop_duplicates(
        subset=key,
        keep="last"
    )

    return combined